# QSV2G — Stage 01: Reproducibility Baseline

**Goal:** Verify the exact reference Python source and Section IV snapshot; rerun the original **38 expected-outcome validation cases** and **720 ideal QSS recovery trials**, including **CHARGE/DISCHARGE × AB/AC/BC**; then compare the run to an archived prior result. Export a Kaggle ZIP containing evidence and execution environment.

**Source:** [QSV2G GitHub repository](https://github.com/shanmukhdatta/QSV2G). The Python source and earlier Section IV `.tex` used by the original experiments are embedded in this notebook with SHA-256 checks. The uploaded current manuscript may have changed since that snapshot.

**How to run:** Upload the `.ipynb` to Kaggle → Internet **ON** for package installation if needed → Accelerator **None (CPU)** → **Run All** → download `qsv2g_stage01_baseline_results.zip` from Output. A GPU is not needed. The notebook uses actual ML-KEM-768/ML-DSA-65 through `pqcrypto` or OpenSSL 3.5+; it does not silently substitute classical toy algorithms.

**Scientific scope:** Ideal 27-dimensional qutrit simulation; tests of expected software behavior, *not* physical quantum hardware or formal/malicious-secure QSS proof. A “matched expected outcome” includes deliberately reproduced known limitations. Random session credentials are generated using OS randomness, so complete run outputs are not byte-identical; we compare reproducible outcomes and structural evidence, not ciphertext bytes.

**What is new versus the prior final-validation notebook?** A packaged reference result fixture, pinned code/paper SHA checks, direct archived-case comparison, a stronger evidence manifest, baseline-specific CSV/JSON exports, and ZIP validation.

## 0 — Environment setup and reproducibility configuration

Record actual package and backend versions. CPU-only protocol functions are used; the notebook reports the run timestamp and available accelerator type. Dependencies may be installed into the notebook session. No experiments are claimed to have run until their cells execute.

In [ ]:
import os, sys, subprocess, importlib.util, pathlib, platform, time
# Only the missing runtime packages are installed; the validated reference source stays unchanged.
packages = {'numpy':'numpy', 'pandas':'pandas', 'cryptography':'cryptography', 'matplotlib':'matplotlib'}
missing = [pkg for module,pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
# pqcrypto provides PQC when local OpenSSL lacks ML-KEM/ML-DSA support (common on Kaggle).
openssl_ver = subprocess.run(['openssl','version'],capture_output=True,text=True).stdout.strip() if __import__('shutil').which('openssl') else 'not installed'
if importlib.util.find_spec('pqcrypto') is None:
    if ('OpenSSL 3.5' not in openssl_ver and 'OpenSSL 3.6' not in openssl_ver):
        subprocess.check_call([sys.executable,'-m','pip','install','-q','pqcrypto'])
print('Python:',sys.version.split()[0], 'OS:', platform.platform())
print('OpenSSL:',openssl_ver)

import csv, io, hashlib, base64, zlib, re, shutil, datetime, zipfile, json
from pathlib import Path
RUN_STARTED_UTC = datetime.datetime.now(datetime.timezone.utc).isoformat()
STAGE_ROOT = (Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd())/'qsv2g_stage01_baseline_results'
OUT = STAGE_ROOT/'validation'
OUT.mkdir(parents=True,exist_ok=True)
print('Run started (UTC):', RUN_STARTED_UTC)
print('Baseline output directory:',STAGE_ROOT.resolve())


## 1 — Reference materials and SHA-256 checks

The source is copied from the published research repository snapshot, not downloaded dynamically. Both the implementation and Section IV reference are verified before any tests run. The prior validation ZIP serves as a **comparison fixture**, never as a replacement for newly executed results. It is the user-supplied archive from the earlier run, not independent proof of its execution environment.

In [ ]:
REFERENCE_CODE_SHA256 = '2c86ca9eecbc5925100722119f061e640df03f993e14909b3078b23406330758'
REFERENCE_TEX_SHA256 = '4ca43b1f6debd2e2bf314ff9adf74f0ad08ba1702e1055eca1943b9533290ef2'
ARCHIVED_VALIDATION_SHA256 = 'f05cbc3eb70e3425438a22cae25d292ce3ce78787052c1b8e1e7d375d3462030'

PINNED_TEX_B64 = """...
OR2NRqButukfX+KNzD7cfT1uqa/F/wBHfjBB"""
ARCHIVED_VALIDATION_B64 = """eNp9WHc0nFv3HmWIPhJRRr9Ej15C9N4JgpGMwZgwupnRCSJK1CjRJdFCdCaI3k2QIFp0QkRESxDCKJ977/dbWfH77n3fdf44
e73nOfusvd9n7+cY6xER0wEAgEuAj2bmUKswSkYtQgCgmwgAYDi3etu6IO1tMUh3Nxjay9XVFuUn7IR2d/MynXGaFQUFDVlQ
4seSfBNXBqb7THNzh934bHMrVVO1ILR3UNx+WL03bT4uDlmvjEUtJdCYz3kSIgyA0o6mLynr5h+qGhdOlhbMLbABe2EUN/xI
Ex6lOm0skekrYSbz48TBDiqyzPpRjuNkBa65Ei1+FBmpxprMUlkUyG5QOCo/rjvrNpYmgagkqtwDblaB4/BnGmeb8XmT7zVk
3owmfW9n1PYFke15SBlArmJsPvVIBSySJ1NDQTfnsW90V3abfy2S8Qe4Z5TPlSUg9xgZZShpopDnukorf12xyqyd4KNb1vHr
Fj1pBDAt+bvXwTX+mMK+Rc1K42MFftr9JIaV7p7btwUgXCWaV6VETHqls4uQ5ZAMuwysOVb/g1cE0hrb2Df5wsk3FSueD791
HSLPpv/8pYb8/cXBJ2+xPOOIGIOBRBk5PXS589RXg4oMF8j4G0zXjRdlfMZ9r3gCofZMlUxOrR75CozbG2byia2PnTOH+bfW
WPl16gzX7jQWhjAPR4iur/Ky+jRdt3fWZHzDGJhGf2NZNnA6s1cvuDHiLEzxGX+koeZmoJKnWxxrgk6DYamXNZQHV0KHeCqf
9dqITW6Ur3Xw61VoADQiwWmP56m6sdkHzAJceW2c7cGa1pLXKUlZDMfHWKAVQZH787ScW71xBNRfdBpq1kBbPk1Mhv3XZh7q
pLvTgBy5HjxfzF7dN8CJlv7o4iFOv0qDknvbuG74XI6JcIXj+BVg/q3umjLzDBgPqkgFDxWCNoxqafL3Cky7q6yhzackxr/l
nhLrvV3i87wTugQAsJxbEZ5ef2cexh0Gd7dHwFxtPTyQbg7CcLT3k9szhsxioKBti59EzWbRxOXTRZLGypIlHLSVgribfVa+
m3E4aiHYQnVLAyhJGXbp5n52Q47ERGeXHtkAR3NNROmL8eGQ72vY7ooMyLuZRauY9kJo6BWKxSnrzhjVBJjNIZF5YZiGLFG5
H5Akl5jLFYgAfo88yyOakHMM13ujnx5NFY1I5x96nNdBXlAff00GQT+j/3zJlMGGN5qB4r6SH8vdZ+8ZEID4hdjI/AdJpmjg
Vb33zVdI7qwfq/A9ItMlhm3DDVDfhZiPAu84bB5zjFyuvkuynSIi06d1pyTayJnU+BP6hQBILi2gcfjLAvkKH25sN/7grIxX
PKgOONFJkGzFIaDzOqWJtgJ+SD7zmo26CizUz6g5/eKWIhuaaKjvE8DV4/JPxzAoRdnU4Dhcp2j60qjNRmEi65BY3UJ9NanG
FsFJDn+5dcdHo3tlDnHDugLMjxXmjXiyJ5eeDDftJhAJ7oruTOaF7jwXejkTHGdOMjnOlVw7DH5L2KpLersZdJObuj7rzTf+
u3vsyZz7P8gy3wDf29GQVJuUVwnTFudzv8rdCxUHn/FvVfT21MdsC90kbuHt9llUIASuxefBBDLvd+eCZHzS2vIZP6OvMcl+
JHMB05UTtY0Sro85zwPXsYmiE9wme6nq8Veen4xZX6H6w2/+Ob5Cv3HTN9uqtQM8zZSpIAhru81JaLDLiLcru853ne2Jvyf3
9Ji3C9y+/NqSF1JQ1hKS0fWu1nLxJr1bLZZXN29DTSQH62vPt83xcp4jloZKTcUv+ZNpernb2Z3iwjIVRVaqg8ffHWkqHlIP
O8s0ZoODLofUG1VtpatSDt9xnFMvsTaM6OmRvTKvJhJuGvVK02pGCVjhfOLNWMBudPzQCR1JU8nTs3ic95otbqrXN8lUAT+x
Kx+djXwSBcY/JT96zfqNUJx7/qY/y8vZ2omknJ05CM3Vmdn4mHd+dfuHlPetwl+TBjtHk4CZ62VpNdVoQUeRXC3OAl67ohJr
rI4pfXmb+GrXkyOPtgesReraFi1Gp+y//z2uSVhKZwIAQPWcvS+fW000VNQNNGAmGqa39c1MhTG+mPDEDrcZUcqIbX9yReVl
wqZt41cQMXvWZh6+CCi55j07t6T0WylE22tavD1VOZ9jqGKUDoPG0jYHmrR+tgOvNNssEcQVtkuMp5ZFHJwFdUp8BIw+++Ls
Kps/OHfwNajqGE1fPVJqHeBTbehhUz4RbpCIDu3O2DSxJHhic1ILW9ieoOF4l2dfYsMFNpaa5a3gMos2mLo3qSdIJGDaQqbR
bL8Uqlvg3K75kthUL5zuIHuyGTT4iSuv2EQS3Dhcmq6uI5h8OyWUY3mte+XShuRi2PBAOx0XNOttlccwldAVmrwpixuJO6HL
GnkUPvwdQxE9VhEy6zGTKtfwTu2cCrsNK/dJ0sQzszXoBFZTh9saq5ihnXFBe2tUICuvQGgem1Uz6enKqtEz86wfo+VtbHU+
omKNNzrYlml27iwSBi/oM9QNwJjn4qOrTYPwk2Fks/h+NjyrzeU0JWxcvFwXSQJOXbmEmpG/0D/rk2XTN+bfQwNlG5Z0OZ9V
cP8dGk80GuZhi0TBkPYIW5c/6WxzopoqhAOk/H4ewnFLBmDhQRBmYWlriLlfLSbadXL81fXb0Kaxj44Jv+G1FS0z77Qa8xTO
n196p0vqAYB2MyJACA2aCgB4AJuGAACMWhIUAMDyl3sEABv69zJYAEA0IZgeAPj8/h0hoF3I+nffBizfFssAAQD2cxPz7wUf
botGwFAItJcLBv2nk+OWBugxTVDLqgXm0K54xG6JxQTIWy70wBSYDl0WlBL84c+beq87cFKoTbFSa8d/ZcdOlVa6IcTl3V7A
7gu814Di13ZIobzDccOygEfOVLiDT1ykod8z5JVJr/gw8ulMobKbp9eGDJ3NqVZG85tvZNLv8ZMUEclWbEg1eaUyaPh5Wz+d
VchuejO1jRtai/oEOy0rUZbTYPJG7k+JZ+ramBHFhh3cygDdaTQ3sN/x/BhGfWs+2cVeS9ZL96Y/Kp39HuaGQqD/43f9SgOq
IBkasqOTwKchsTTC/WIva1xL1QZ4jEuDSlVrj3A/RxWWGQJ62ddhG5CzZwmwDabg/DPM0NE2WdtPme8xTcCZSSXQxzgnTKHr
Xv2q56xUiZ2I1oSUX7pwsrjU2mLffHr54yb5OhaXYyD/zqwQb6c2nkOIXoxSO9kJocBcVmyzWaDAJHh6YjGNT3IaOKmUI3XX
LN5P/Up8Jw/e+Hh3yRUnXPYMYUGypygN3jDJf4zPIq0djjvtHDQtjH1Y2qGeNVB8Ej1pVhehPyRxJMW685MjAL91vKzjNz8g
d8TJKuwOJhfEyV/iJnArK1Uop3OMuDqiXqs/PdZogiK93xVqCavRafgEyyMAFmlPD5qY6xqqEOOe5XnfVfJ4ivT+JkDIpkVd
V0N1tXJ3iv3a0VQdfus13UcqnjTQEyU1JdsI9KRvQ16hprgypXfa+iPux6VAYshi7hh2M9YvRa0aRj0a5HlDNueMoWKZAIy3
WF6F9js7T5KS70cXZspvNNlZfRvpFLFhWPiSbRLzTY+AERfskClFEGbGW/fwoMDOEhL7dB1anRYYNkIjiOCbnewIFvXf5zRb
xoHQxK6WLvSVKqos6rClnwV3buTgagA5zVXl7FoiIiRS3FHkS1nT3OvgrqdhYLeKR7UsJVp7rywHiIX7E8lDNnOi3CSN4a95
p+AnIVXDkTooSclQaVcDAthsdZZlImOZVq9vJVMgeHqH8YxgQKjmsjMe/3B+zXHd25Y5QXuj8FFL39BgO+anCK2iIo97vtrn
b1X4L+aCjvaURVqYlKsWFiOl3dhcfr0aWbcnvXpIS54XCTHO72ToUtrACMHXGf6XNx7GaeePWIenP/3WMvrHZhuPkzWQbP7A
YUMzclPnI1d/Y0Fdod19sQfyiY/y4GvBY0XNgQ4vcQ7RKtTjIbYtbPsEXXK03bjUdbFBUq4sMAQLaSaTwCxpyFUZNZhvdypJ
JdO/XJqQFFxd2ciTZxpsNkmizaJpjTtoVj+DosCSIWEHXnSfP+jKvKQZKBbNOyJ3v9x061AkT+K+2mGkwV0UUYv1Ai/Ox1qL
OgtQaPvHGt2I9F7+vOFyMfxDwY/LV7EMQmaWq4j7hHOJaweg5vonqd+SrYA/DHadogu3ih9t6c2SuNiwBPYWF4dkkoZ8Et4J
InPXVsTIoKDV2u+rOwsE2jYgg/1Y8e3qyMhqfpWqwtbYFlrO0la+yL5xvYCOoXrtqM2X4UGJtgV3giGKz8vHFCgawobV9smO
QphzLpVQ+gR1lCiOumnNQwVCbdrL+MKvB+D63+Klh+txuX739YwZXeLWbu8Y3r7qozbUbNHdw13kGDGtYupfkyKc4HngNDWZ
VdT0gF07VTeIzWgDtTPjHU55Lb3wKEVPl/YEDCO1W42LwIcqZoZophkQ2wvjuvamVOqgPewM0plWDV5GmKNgTqvAJ1+o68/W
1HERiaeEq3Zp0/3m7nJBAdaD99gvD22udh82w9JPG3sb9Vpu7knVW/K+kNrxe1O3qcp8uLvwO98e1DsMa53POs8H+HzYemEc
3VFI//9SrrsbBuXu8hfdDogM82u/FRrR0Tc0vM6nY/CeX1hH32gFte9BifIZ9xzzGiHjFzAYJ/Ok2vmxh6b09FLRll1OQnX+
sbLEFIvTt4cE+ldnUAJ+357eqDOd9rxDSDzvsxn/7LF9PVyQcCQGZos+13cefzrx1+YIEwP3816hBW8BahBu3ajpMkSu2xIk
1TmP2NaHJq6pFws10tW+xf/In50AZmZxR21/D7LRbWAfQ9/l0XiH5H/4CF7b759XPJ9rQhEPX51B4QAJH2pRJ75UiXtvd4i3
HoqZaA9szhlgkW6p65Y0nfEmhE+yeYgpCaiz4TFL/EHhb3abxZqKnDyDwcnfgb6DjFIH26+HTYfECXvprUxQ/XYAxfd2fgH1
Zy3z8+6Zrugmma0hmNNG5KJUPXHfgazN4QnZCKNBEr944NXKHyIctKnELF2W5Z8Wbw9bJiXglKUyiVzONBfLfLmNqbkZWQ2y
37m1NU6X+SzEjbJADAamPpa0H2pz7GW3pO91Oa7GP3Sha2xfdYrpuDkkO/7oLv0DJe7WXd9LxEreTRTHOUsdpGcJLlOcn+z9
9lo+zBPqhEhvw/uhDun7JlJKm/MjytXc9L0WGF1stSq9Y43jbMqxFgVDhOiQpXrV3JjMTdNi3cJr0vAMdfN2ppGdz263oBQv
TBUm8+btEpzaop0cOpynWtO5U27EtJA4JXAz7NFgIeGxIafgCT1dnrL7a6pS9YhFCTKWnnoPVeRQT4Lq7kTzPDkX/Rpw6KnW
c20UizNWLTxsBW/ONGpZvgB9GF8oI867/DyQjsXQnPtK+caN5sYVj7rdVr+f3gdV16c8i0YNe27qDQq49pJRERhwdo6FjNT0
dXxV+j2NUrZMnGLPZ5XnDSfTfzsaNMbWDumC9EegYHBHBNz5rzwqT6gm71QGqeMDwwusE+ZaZt5t1kex3KKhdCT0oU+qlJl+
VAAS76HbjcaZMRahIh1KEk4lZyNk01L0Msjfto1KfjarDOpbExfykF+d5njctS36XC0qHlS4TogDkbV8UuB/RYrqdekHj2uJ
6AW1b7WdTEq/umdeFPVi27KK2stZojJLozmKmEf6yszxLsWFfpnx0OL9uftiRH+rTQ+UO8Yd7u4CQ/gi4F5/606ULRzx5ymg
JgboaVFQy65g+t0mX6G+uu5ExhebkpIZqkNgyLwN1Wk09+gwDMakHVOabO7vvrDVuim4Asq2uO8RUvuVNbXG09gGOmDNtG6f
YG63HqeklrtHO3M5flLb9Yi8HOcjEoPVIh1Mqc/amh4IeAMN/Rowk1P1UIzZcoyzipOSQ0by49LbXKrLjcxWwWSq1+ndX+qO
9WZUACnTuyDSbQLfNNlZMSdvlzpuuD9hJxApnaNmjFHvDDGm7RGVlXC9uWo/JdX/yoSqFRww4jN7w447/MySeo5PRCy2uDiu
NFMZ3y4SDviy8ErhoDUrsvEd2epMUU/9O+8OCbEzSZL1OafEQmU7sQn1Gh1bbMKtl2EazUMGsk0cb8TvDaSMpuQtPeCRFGTk
/SM/dl3oPT+cLvnBC6DfuPz7gYKoNLEP/IVlNnKmPlbw0Rwqxf7kEgNZZA7IU0OhbJF2JzK4rF5UCgbB3LZIs/AFf09ZOGVH
T84QcAaftuIb5sqKrNpE/hjxsXU9lUv53iQ2KpogxCXTds+38QHC8vKu3PdIdVX9Y8jK4WdEsC00jWjhFk1yhbhYdXL3R3VP
fdtXtideacGQk5/NGX6v4pCVVoe7+ytILE0OMfeznbCxc+lEQEhH9M/XXv/3FIQC/vkS7CLIxfuLXyCBhP9+m3ER6aKW+4XU
CPxfyu7i+ouC49f626T/S35cXH9RFPxa/530XyXCRaCL1e4XkC3Vv9W+izgXy9YvnEWqfy5iF1EustYvFF7Qv3DY/4vOBeb4
BfMV9O88YqwHJPnzS7Lz1+f8OD/+Cs5/AGejobs="""

def unpack_pinned(label, b64_string, expected_sha):
    payload = zlib.decompress(base64.b64decode(''.join(b64_string.split())))
    actual_sha = hashlib.sha256(payload).hexdigest()
    assert actual_sha == expected_sha, f'Pinned {label} SHA mismatch: {actual_sha} != {expected_sha}'
    return payload

TEX_BYTES = unpack_pinned('reference LaTeX',PINNED_TEX_B64,REFERENCE_TEX_SHA256)
REFERENCE_TEX = TEX_BYTES.decode('utf-8')
REF_ARCHIVE_BYTES = unpack_pinned('historical results',ARCHIVED_VALIDATION_B64,ARCHIVED_VALIDATION_SHA256)
REFERENCE_ZIP = zipfile.ZipFile(io.BytesIO(REF_ARCHIVE_BYTES))
assert REFERENCE_ZIP.testzip() is None, 'Reference validation archive failed ZIP integrity'
archived_summary = json.loads(REFERENCE_ZIP.read('validation_summary.json'))
assert archived_summary['source_implementation_sha256'] == REFERENCE_CODE_SHA256
assert archived_summary['revised_tex_sha256'] == REFERENCE_TEX_SHA256
print('Pinned Section IV source hash:',REFERENCE_TEX_SHA256)
print('Prior submitted archive hash:',ARCHIVED_VALIDATION_SHA256)
print('Archived baseline: tests',archived_summary['test_count'],'ideal recoveries',archived_summary['qss_ideal_correct'])


## 2 — Unmodified reference implementation, cryptographically pinned

The following cell contains the complete, exact original source (human-readable Python in the string) and executes it **only after** verifying its SHA-256 hash. It includes Bell-state registration, the QSS simulator, ML-KEM/ML-DSA adapters, session protection, and the V2G workflow. **Do not edit this cell** when reproducing the earlier protocol.

In [ ]:
import hashlib
CORE_SOURCE = r'''"""QSV2G Section IV reference prototype (research simulation, NOT deployed secure system).
Source: QSV2G_main8_SectionIV_Revised.tex.
Quantum: exact 27D three-qutrit statevectors with sampled pair measurements.
PQC: actual ML-KEM-768 / ML-DSA-65 via pqcrypto OR OpenSSL >= 3.5.
This is not malicious-secure verifiable QSS. Authenticated channels, secure enrollment,
quantum memory and two-authority honesty are simulation assumptions.
"""
from __future__ import annotations
import base64
import hashlib
import hmac
import json
import math
import os
import secrets
import shutil
import subprocess
import tempfile
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any
import numpy as np
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

LABEL = 'QSV2G-REFERENCE-2026'
AUTHORITIES = ('A','B','C')
SLOT = {'A':0,'B':1,'C':2}

class ProtocolAbort(Exception):
    pass

def require(condition, explanation):
    if not condition:
        raise ProtocolAbort(explanation)

def json_obj(o):
    if isinstance(o, bytes): return {'__type__':'bytes', 'base64':base64.b64encode(o).decode('ascii')}
    if isinstance(o, tuple): return {'__type__':'tuple', 'items':[json_obj(x) for x in o]}
    if isinstance(o, list): return [json_obj(x) for x in o]
    if isinstance(o, dict): return {str(k):json_obj(v) for k,v in sorted(o.items())}
    if isinstance(o, (str,int,bool)) or o is None: return o
    raise TypeError(f'Unsupported field {type(o).__name__}: floating-point values are intentionally disallowed')

def canon(x):
    return json.dumps(json_obj(x), sort_keys=True, separators=(',',':'), ensure_ascii=False).encode('utf-8')

def parse_canon(raw):
    def restore(x):
        if isinstance(x, list): return [restore(y) for y in x]
        if isinstance(x, dict):
            if x.get('__type__')=='bytes' and set(x)=={'__type__','base64'}:
                return base64.b64decode(x['base64'],validate=True)
            if x.get('__type__')=='tuple' and set(x)=={'__type__','items'}:
                return tuple(restore(y) for y in x['items'])
            return {k:restore(v) for k,v in x.items()}
        return x
    return restore(json.loads(raw.decode('utf-8')))

def encode(*fields):
    """Fixed ordering, deterministic, typed-length-separated encoding."""
    out=bytearray()
    for f in fields:
        chunk=canon(f)
        out += len(chunk).to_bytes(4,'big')+chunk
    return bytes(out)

def H(*fields): return hashlib.sha256(encode(*fields)).digest()
def mac(k, *fields): return hmac.new(k, encode(*fields), 'sha256').digest()
def ct_eq(a,b): return hmac.compare_digest(a,b)
def hkdf_extract(salt, ikm): return hmac.new(salt, ikm, 'sha256').digest()
def hkdf_expand(prk, info, n=32):
    out=b''; t=b''
    for i in range(1,math.ceil(n/32)+1):
        t=hmac.new(prk, t+info+bytes([i]), 'sha256').digest(); out+=t
    return out[:n]

def digest_for(label, *items): return H(label, *items)

# ---- Quantum statevectors ----
I2=np.eye(2,dtype=np.complex128)
X2=np.array([[0,1],[1,0]],complex)
Z2=np.diag([1,-1]).astype(complex)
H2=np.array([[1,1],[1,-1]],complex)/np.sqrt(2)
CNOT=np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]],complex)
BELL=np.array([1,0,0,1],complex)/np.sqrt(2)

def dense_encode_decode(two_bits):
    a,b=two_bits
    state=np.kron(np.linalg.matrix_power(Z2,a)@np.linalg.matrix_power(X2,b),I2)@BELL
    final=np.kron(H2,I2)@CNOT@state
    probabilities=np.abs(final)**2
    idx=int(np.argmax(probabilities))
    require(np.isclose(probabilities[idx],1,atol=1e-12),'Bell decoding failure')
    return (idx//2,idx%2)

def dense_transport_tag(tag):
    recovered=bytearray()
    bits=''.join(f'{byte:08b}' for byte in tag)
    out=''
    for k in range(0,len(bits),2):
        aa,bb=dense_encode_decode((int(bits[k]),int(bits[k+1])))
        out+=str(aa)+str(bb)
    recovered.extend(int(out[j:j+8],2) for j in range(0,len(out),8))
    return bytes(recovered)

OMEGA=np.exp(2j*np.pi/3)
X3=np.roll(np.eye(3,dtype=complex),1,axis=0)
Z3=np.diag([OMEGA**j for j in range(3)])


def qss_state(s):
    """|psi(s)> = sum_r |r,r+s,r+2s> /sqrt(3)."""
    require(s in (0,1,2),'Invalid trit')
    state=np.zeros((3,3,3),dtype=complex)
    for r in range(3): state[r,(r+s)%3,(r+2*s)%3]=1/np.sqrt(3)
    return state

def reduced_density(psi, kept):
    moved=np.moveaxis(psi,kept,0).reshape(3,-1)
    return moved@moved.conj().T

def measure_pair(psi, pair, rng, x_fault=False):
    """Sample joint computational measurement of two shares from a 3-qutrit statevector.
    x_fault applies a qutrit-X error to the SECOND selected share before measuring.
    """
    u,v=[SLOT[p] for p in pair]
    state=np.moveaxis(psi,(u,v),(0,1))
    probs=(np.abs(state)**2).sum(axis=2).reshape(-1)
    idx=int(rng.choice(9,p=probs/probs.sum()))
    xu,xv=divmod(idx,3)
    if x_fault: xv=(xv+1)%3
    return xu,xv

def recover_symbol(pair,xu,xv):
    denom=(SLOT[pair[1]]-SLOT[pair[0]])%3
    return ((xv-xu)*pow(denom,-1,3))%3

def qss_consistency():
    Xa=np.kron(np.kron(X3,X3),X3)
    Za=np.kron(np.kron(Z3,Z3),Z3)
    vals=[]
    for s in range(3):
        psi=qss_state(s)
        psi_flat=psi.reshape(-1)
        vals.append({
            's':s,
            'norm':float(np.vdot(psi_flat,psi_flat).real),
            'x_stabilizer_error':float(np.linalg.norm(Xa@psi_flat-psi_flat)),
            'z_stabilizer_error':float(np.linalg.norm(Za@psi_flat-psi_flat)),
            'one_share_distances':[float(np.linalg.norm(reduced_density(psi,b)-np.eye(3)/3)) for b in range(3)],
        })
    return vals

# ---- Actual standardized post-quantum cryptography, no silent fallbacks ----
class PQCrypto:
    def __init__(self):
        self.kind=None
        try:
            from pqcrypto.kem import ml_kem_768
            from pqcrypto.sign import ml_dsa_65
            self.kem=ml_kem_768; self.sig=ml_dsa_65
            self.kind='pqcrypto native ML-KEM-768 + ML-DSA-65'
        except (ImportError,ModuleNotFoundError):
            exe=shutil.which('openssl')
            if exe:
                test=subprocess.run([exe,'list','-kem-algorithms'],capture_output=True,text=True)
                sig=subprocess.run([exe,'list','-signature-algorithms'],capture_output=True,text=True)
                if 'ML-KEM-768' in test.stdout and 'ML-DSA-65' in sig.stdout:
                    self.kind='OpenSSL 3.5+ ML-KEM-768 + ML-DSA-65 (subprocess)'
                    self.openssl=exe
            if self.kind is None:
                raise RuntimeError('Real ML-KEM-768/ML-DSA-65 unavailable. Enable Kaggle Internet and pip install pqcrypto. No classical substitute will be used.')
    def _openssl_call(self, arguments, input_files=None, output_files=()):
        with tempfile.TemporaryDirectory(prefix='qsv2g_pqc_') as folder:
            root=Path(folder)
            for name,data in (input_files or {}).items(): (root/name).write_bytes(data)
            argv=[self.openssl]+[str(root/(v[1:])) if isinstance(v,str) and v.startswith('@') else str(v) for v in arguments]
            p=subprocess.run(argv, stdout=subprocess.PIPE, stderr=subprocess.PIPE, cwd=root, timeout=30)
            if p.returncode:
                raise RuntimeError('OpenSSL cryptographic call failed: '+p.stderr.decode('utf-8',errors='replace')[:500])
            return (p.stdout, *[(root/x).read_bytes() for x in output_files])
    def kem_keygen(self):
        if self.kind.startswith('pqcrypto'):
            pub,priv=(self.kem.keygen() if hasattr(self.kem,'keygen') else self.kem.generate_keypair()); return pub,priv
        # key generation and public-key extraction share one temporary directory
        with tempfile.TemporaryDirectory(prefix='qsv2g_kem_') as d:
            sk=Path(d)/'sk.pem'; pk=Path(d)/'pk.pem'
            subprocess.run([self.openssl,'genpkey','-algorithm','ML-KEM-768','-out',str(sk)],check=True,capture_output=True)
            subprocess.run([self.openssl,'pkey','-in',str(sk),'-pubout','-out',str(pk)],check=True,capture_output=True)
            return pk.read_bytes(), sk.read_bytes()
    def encaps(self,pub):
        if self.kind.startswith('pqcrypto'):
            return self.kem.encaps(pub) if hasattr(self.kem,'encaps') else self.kem.encrypt(pub)
        _,ct,ss=self._openssl_call(['pkeyutl','-encap','-pubin','-inkey','@pk','-out','@ct','-secret','@ss'], {'pk':pub},('ct','ss'))
        return ct,ss
    def decaps(self,priv,ct):
        if self.kind.startswith('pqcrypto'):
            return self.kem.decaps(priv,ct) if hasattr(self.kem,'decaps') else self.kem.decrypt(priv,ct)
        _,ss=self._openssl_call(['pkeyutl','-decap','-inkey','@sk','-in','@ct','-secret','@ss'],{'sk':priv,'ct':ct},('ss',))
        return ss
    def sig_keygen(self):
        if self.kind.startswith('pqcrypto'):
            return self.sig.keygen() if hasattr(self.sig,'keygen') else self.sig.generate_keypair()
        with tempfile.TemporaryDirectory(prefix='qsv2g_sig_') as d:
            sk=Path(d)/'sk.pem'; pk=Path(d)/'pk.pem'
            subprocess.run([self.openssl,'genpkey','-algorithm','ML-DSA-65','-out',str(sk)],check=True,capture_output=True)
            subprocess.run([self.openssl,'pkey','-in',str(sk),'-pubout','-out',str(pk)],check=True,capture_output=True)
            return pk.read_bytes(),sk.read_bytes()
    def sign(self,priv,msg):
        if self.kind.startswith('pqcrypto'):
            return self.sig.sign(priv,msg)
        _,sig=self._openssl_call(['pkeyutl','-sign','-inkey','@sk','-in','@msg','-out','@sig'],{'sk':priv,'msg':msg},('sig',))
        return sig
    def verify(self,pub,msg,sig):
        if self.kind.startswith('pqcrypto'):
            try:
                result=self.sig.verify(pub,msg,sig)
                return result is None or result is True
            except Exception: return False  # InvalidSignatureError from pqcrypto 1.x, ValueError on older versions
        try:
            self._openssl_call(['pkeyutl','-verify','-pubin','-inkey','@pk','-in','@msg','-sigfile','@sig'],{'pk':pub,'msg':msg,'sig':sig})
            return True
        except RuntimeError: return False

@dataclass
class Traffic:
    key:bytes
    send_counter:int=0
    seen:set=field(default_factory=set)
    def encrypt(self,pt:bytes,ad:bytes):
        require(self.send_counter<2**96-1,'AES-GCM counter exhaustion')
        nonce=self.send_counter.to_bytes(12,'big'); self.send_counter+=1
        ct=AESGCM(self.key).encrypt(nonce,pt,ad)
        return {'nonce':nonce,'ciphertext':ct}
    def decrypt(self,pkt,ad):
        nonce=pkt['nonce']
        require(len(nonce)==12 and nonce not in self.seen,'duplicate or malformed AEAD nonce')
        try: pt=AESGCM(self.key).decrypt(nonce,pkt['ciphertext'],ad)
        except Exception as e: raise ProtocolAbort('AEAD authentication failure') from e
        self.seen.add(nonce)
        return pt

@dataclass
class QSSResource:
    nu:str
    dealer_secrets:dict
    states:dict
    commitments:dict
    state:str='UNUSED'
    lock:tuple|None=None
    def reserve(self,ac,pair):
        require(self.state=='UNUSED','QSS resource already reserved or consumed')
        require(len(set(pair))==2 and all(a in AUTHORITIES for a in pair),'pair invalid')
        self.state='RESERVED'; self.lock=(ac,tuple(pair))
    def reconstruct(self,ac,pair,rng,tamper=False):
        require(self.state=='RESERVED' and self.lock==(ac,tuple(pair)),'QSS reservation mismatch')
        recovered={a:[] for a in AUTHORITIES}
        try:
            for dealer in AUTHORITIES:
                for k,psi in enumerate(self.states[dealer]):
                    xu,xv=measure_pair(psi,pair,rng,x_fault=(tamper and dealer=='A' and k==0))
                    recovered[dealer].append(recover_symbol(pair,xu,xv))
            for dealer in AUTHORITIES:
                expected=H('QSV2G-QSS-COM',dealer,self.nu,recovered[dealer])
                require(ct_eq(expected,self.commitments[dealer]),'QSS reconstructed commitment mismatch')
            combined=[sum(recovered[a][k] for a in AUTHORITIES)%3 for k in range(len(recovered['A']))]
            return combined
        finally:
            self.state='CONSUMED'
            self.states.clear()  # simulated irreversible measurement / erasure

def new_qss_resource(L=162,rng=None):
    rng=rng or np.random.default_rng(2026)
    nu=secrets.token_hex(12)
    dealer_secrets={a:[secrets.randbelow(3) for _ in range(L)] for a in AUTHORITIES}
    states={a:[qss_state(s) for s in dealer_secrets[a]] for a in AUTHORITIES}
    commits={a:H('QSV2G-QSS-COM',a,nu,dealer_secrets[a]) for a in AUTHORITIES}
    return QSSResource(nu,dealer_secrets,states,commits)

class QSV2G:
    """Single-session strict reference implementation with explicit trusted authority simulation."""
    def __init__(self,crypto=None, seed=2026,L=162,available=None):
        self.pqc=crypto or PQCrypto(); self.rng=np.random.default_rng(seed)
        self.available=set(available if available is not None else AUTHORITIES)
        self.L=L; self.log=[]; self.replay_ev=set(); self.replay_cs=set(); self.executed=set()
        self.wire_bytes={}  # Message sizes include ciphertext/tag and 96-bit GCM nonce; exclude implied AAD.
        self.reg_ev=secrets.token_bytes(32); self.reg_cs=secrets.token_bytes(32)
        self.pid_master=secrets.token_bytes(32); self.id_ev='ev_internal_reference_001'
        self.csid='CS_DEMO_01'; self.op='CHARGE'
        self.sigkeys={a:self.pqc.sig_keygen() for a in AUTHORITIES}
        self.resource=new_qss_resource(L,self.rng)
        self.t0=int(time.time()); self.stage='NEW'; self._last_mark=time.perf_counter()
    def mark(self,phase, status='PASS', **details):
        now=time.perf_counter()
        self.log.append({'phase':phase,'status':status,'elapsed_ms':round((now-self._last_mark)*1000,3),**details})
        self._last_mark=now
    def register(self):
        require({'A','B'}<=self.available,'UC-A and UC-B required for online registration')
        rA=secrets.token_bytes(16)
        ev_tag=mac(self.reg_ev,'REG',self.id_ev,rA)
        # 32B tag -> 128 Bell-pair encodings, decoded with 2-qubit simulator
        require(ct_eq(dense_transport_tag(ev_tag), mac(self.reg_ev,'REG',self.id_ev,rA)),'EV Bell verification')
        self.pid=mac(self.pid_master,self.id_ev,secrets.token_bytes(16),self.t0+86400)[:16].hex()
        rB=secrets.token_bytes(16)
        cs_tag=mac(self.reg_cs,'CSREG',self.csid,rB)
        require(ct_eq(dense_transport_tag(cs_tag),mac(self.reg_cs,'CSREG',self.csid,rB)),'CS Bell verification')
        self.stage='REGISTERED'; self.mark('registration', bell_pairs_per_entity=128)
    def authenticate(self, tamper_ev=False, tamper_cs=False):
        require(self.stage=='REGISTERED','register first')
        require({'A','B'}<=self.available,'UC-A and UC-B required for online participant validation')
        self.ne=secrets.token_bytes(16); self.n_c=secrets.token_bytes(16)
        self.ts_e=self.t0; self.ts_c=self.t0
        self.ev_tag=mac(self.reg_ev,'EV_AUTH',self.pid,self.csid,self.op,self.ne,self.ts_e)
        submitted=self.ev_tag
        if tamper_ev: submitted=bytes([submitted[0]^1])+submitted[1:]
        require(abs(int(time.time())-self.ts_e)<=300,'stale EV timestamp')
        require((self.pid,self.ne) not in self.replay_ev,'EV nonce replay')
        require(ct_eq(submitted,mac(self.reg_ev,'EV_AUTH',self.pid,self.csid,self.op,self.ne,self.ts_e)),'EV proof rejected')
        self.replay_ev.add((self.pid,self.ne))
        cs_tag=mac(self.reg_cs,'CS_AUTH',self.csid,self.pid,self.op,self.ne,self.n_c,self.ts_c)
        if tamper_cs: cs_tag=bytes([cs_tag[0]^1])+cs_tag[1:]
        require(abs(int(time.time())-self.ts_c)<=300,'stale CS timestamp')
        require((self.csid,self.n_c) not in self.replay_cs,'CS nonce replay')
        require(ct_eq(cs_tag,mac(self.reg_cs,'CS_AUTH',self.csid,self.pid,self.op,self.ne,self.n_c,self.ts_c)),'CS proof rejected')
        self.replay_cs.add((self.csid,self.n_c))
        ua_conf=mac(self.reg_ev,'CS_CONF',self.pid,self.csid,self.op,self.ne,self.n_c,self.ts_c,True)
        require(ct_eq(ua_conf, mac(self.reg_ev,'CS_CONF',self.pid,self.csid,self.op,self.ne,self.n_c,self.ts_c,True)),'EV cannot verify station attestation')
        self.stage='AUTHENTICATED'; self.mark('cross_domain_authentication')
    def establish_session(self):
        require(self.stage=='AUTHENTICATED','authenticate first')
        ek,dk=self.pqc.kem_keygen()
        bind=encode(self.pid,self.csid,self.op,self.ne,self.n_c,self.ts_e,self.ts_c,ek)
        proof=mac(self.reg_ev,'KEM-EV',hashlib.sha256(bind).digest())
        require(ct_eq(proof,mac(self.reg_ev,'KEM-EV',hashlib.sha256(bind).digest())),'KEM public key not authenticated')
        ct,ss_cs=self.pqc.encaps(ek)
        T=encode(self.pid,self.csid,self.op,self.ne,self.n_c,self.ts_e,self.ts_c,ek,ct)
        transcript_hash=hashlib.sha256(T).digest()
        cs_proof=mac(self.reg_cs,'KEM-CS',transcript_hash)
        require(ct_eq(cs_proof,mac(self.reg_cs,'KEM-CS',transcript_hash)),'KEM ciphertext station authentication failed')
        ua_conf=mac(self.reg_ev,'KEM-CONF',transcript_hash,'ACCEPT')
        require(ct_eq(ua_conf,mac(self.reg_ev,'KEM-CONF',transcript_hash,'ACCEPT')),'KEM authority binding mismatch')
        ss_ev=self.pqc.decaps(dk,ct)
        require(ct_eq(ss_ev,ss_cs),'KEM shared secrets mismatch')
        self.sid=H('QSV2G-SID',T)[:16].hex()
        salt=hashlib.sha256(self.ne+self.n_c).digest()
        prk_e=hkdf_extract(salt,ss_ev); prk_c=hkdf_extract(salt,ss_cs)
        def key(prk,label): return hkdf_expand(prk, label.encode()+transcript_hash,32)
        k_e2c=key(prk_e,'QSV2G-E2C'); k_c2e=key(prk_c,'QSV2G-C2E')
        require(k_e2c != k_c2e,'Directional traffic keys unexpectedly identical')
        k_ev_conf=key(prk_e,'KC-EV'); k_cs_conf=key(prk_c,'KC-CS')
        mu_e=mac(k_ev_conf,'EV-FINISHED',self.sid,transcript_hash)
        require(ct_eq(mu_e,mac(key(prk_c,'KC-EV'),'EV-FINISHED',self.sid,transcript_hash)),'EV key confirmation failed')
        mu_c=mac(k_cs_conf,'CS-FINISHED',self.sid,transcript_hash,mu_e)
        require(ct_eq(mu_c,mac(key(prk_e,'KC-CS'),'CS-FINISHED',self.sid,transcript_hash,mu_e)),'CS key confirmation failed')
        self.e2c=Traffic(k_e2c); self.c2e=Traffic(k_c2e)
        self.stage='SESSION'; self.mark('session_establishment', backend=self.pqc.kind, kem_ciphertext_bytes=len(ct), sid_bytes=len(bytes.fromhex(self.sid)))
        # Ephemeral key material references are not kept as instance attributes.
    def authorization_context(self, op='CHARGE', limit_mwh=12000, lifetime_sec=600, tamper_ciphertext=False):
        require(self.stage=='SESSION','session not established')
        require(op in ('CHARGE','DISCHARGE') and limit_mwh>0,'invalid operation or energy limit')
        self.op=op
        self.limit={'type':'ENERGY','max':int(limit_mwh),'unit':'mWh'}
        self.vld=(self.t0-2,self.t0+int(lifetime_sec))
        self.ctx=(self.pid,self.csid,self.sid,self.op,self.ne,self.n_c,self.ts_e,self.ts_c,self.limit,self.vld)
        mctx=encode(*self.ctx)
        ad=encode('QSV2G-CTX',self.sid,self.csid)
        pkt=self.c2e.encrypt(mctx,ad)
        self.wire_bytes['context_offer']=len(pkt['nonce'])+len(pkt['ciphertext'])
        if tamper_ciphertext:
            pkt['ciphertext']=bytes([pkt['ciphertext'][0]^1])+pkt['ciphertext'][1:]
        ev_received=self.c2e.decrypt(pkt,ad)
        require(ct_eq(ev_received,mctx),'EV context disagreement')
        require(self.vld[0]<=int(time.time())<=self.vld[1],'context expired')
        self.ac=H('QSV2G-AC',mctx)
        self.consent=mac(self.reg_ev,'AUTHZ-CONSENT',self.ac)
        ack={'sid':self.sid,'ac':self.ac,'consent':self.consent}
        ack_pkt=self.e2c.encrypt(canon(ack),encode('QSV2G-CTX-ACK',self.sid))
        self.wire_bytes['context_ack']=len(ack_pkt['nonce'])+len(ack_pkt['ciphertext'])
        require(ct_eq(self.e2c.decrypt(ack_pkt,encode('QSV2G-CTX-ACK',self.sid)),canon(ack)),'EV context ack failed')
        self.stage='CONTEXT'; self.mark('context_agreement', context_hash=self.ac.hex())
    def authority_verify(self, approvals=('A','B','C'), revoked=False, bad_consent=False):
        require(self.stage=='CONTEXT','context missing')
        require({'A','B'}<=self.available,'UC-A and UC-B online validation required (full system not arbitrary-one-outage tolerant)')
        require(self.vld[0]<=int(time.time())<=self.vld[1], 'authorization expired')
        consent=self.consent if not bad_consent else bytes([self.consent[0]^1])+self.consent[1:]
        require(ct_eq(consent,mac(self.reg_ev,'AUTHZ-CONSENT',self.ac)),'UC-A rejected EV consent')
        require(not revoked,'EV revoked')
        require(ct_eq(H('QSV2G-AC',encode(*self.ctx)),self.ac),'context mismatch')
        self.decisions={a:(a in approvals and a in self.available) for a in AUTHORITIES}
        self.accepted=[a for a in AUTHORITIES if self.decisions[a]]
        require(len(self.accepted)>=2,'fewer than two distinct approvals')
        self.stage='APPROVED'; self.mark('authority_validation', approved=','.join(self.accepted))
    def threshold_reconstruct(self,pair=('A','B'),tamper_share=False):
        require(self.stage=='APPROVED','authority approval missing')
        require(len(pair)==2 and pair[0]!=pair[1] and set(pair)<=set(self.accepted),'not an approved distinct authority pair')
        require(self.resource.state=='UNUSED','resource must be unused')
        self.pair=tuple(pair)
        self.resource.reserve(self.ac,self.pair)
        recovered=self.resource.reconstruct(self.ac,self.pair,self.rng,tamper=tamper_share)
        self.reconstruction_hash=H('QSV2G-REC',self.ac,self.resource.nu,recovered)
        self.stage='RECONSTRUCTED'; self.mark('qss_threshold',pair=''.join(pair),recovered=True, consumed=self.resource.state)
    def issue_token(self,tamper_signature=False):
        require(self.stage=='RECONSTRUCTED','QSS reconstruction missing')
        u,v=self.pair
        statement=('QSV2G-AUTHZ',self.ac,self.sid,self.resource.nu,u,v,self.reconstruction_hash,self.vld,'ACCEPT')
        msg=encode(*statement)
        signatures={a:self.pqc.sign(self.sigkeys[a][1],msg) for a in self.pair}
        if tamper_signature:
            signatures[u]=bytes([signatures[u][0]^1])+signatures[u][1:]
        # UC-B verification; all public keys are trusted/bound to authority identities in simulation.
        require(all(self.pqc.verify(self.sigkeys[a][0],msg,signatures[a]) for a in self.pair),'invalid authority signature')
        self.token={'statement':statement,'signatures':signatures}
        self.stage='TOKEN'; self.mark('authorization_token', signer_count=2, signature_bytes=sum(map(len,signatures.values())))
    def protected_response(self,tamper_response=False, tamper_token_after_collector=False):
        require(self.stage=='TOKEN','no accepted authorization proof')
        token={'statement':self.token['statement'],'signatures':dict(self.token['signatures'])}
        if tamper_token_after_collector:
            signer=self.pair[0]
            sig=token['signatures'][signer]
            token['signatures'][signer]=bytes([sig[0]^1])+sig[1:]
        resp={'sid':self.sid,'pid':self.pid,'csid':self.csid,'ac':self.ac,
              'op':self.op,'ne':self.ne,'nc':self.n_c,'lim':self.limit,
              'vld':self.vld,'decision':True,'auth_token':token}
        ad=encode('QSV2G-RESP',self.sid,self.csid)
        pkt=self.c2e.encrypt(canon(resp),ad)
        self.wire_bytes['authorization_response']=len(pkt['nonce'])+len(pkt['ciphertext'])
        if tamper_response:
            pkt['ciphertext']=pkt['ciphertext'][:-1]+bytes([pkt['ciphertext'][-1]^1])
        received=parse_canon(self.c2e.decrypt(pkt,ad))
        # EV validates independently from received wire bytes, never from a local CS plaintext.
        require(received['sid']==self.sid and received['pid']==self.pid and received['csid']==self.csid,'identity/session mismatch')
        require(received['op']==self.op and received['ne']==self.ne and received['nc']==self.n_c,'nonce/op mismatch')
        require(received['lim']==self.limit and received['vld']==self.vld and ct_eq(received['ac'],self.ac),'transaction context mismatch')
        require(self.vld[0]<=int(time.time())<=self.vld[1],'authorization expired at EV')
        require(received['decision'] is True,'EV received a reject response')
        wire_token=received['auth_token']
        statement=wire_token['statement']
        require(statement[1]==self.ac and statement[2]==self.sid and statement[3]==self.resource.nu,'signed statement context mismatch')
        # The EV cannot calculate or know h_rec; it can only verify the signed attestation.
        require(isinstance(statement[6],bytes) and len(statement[6])==32 and statement[7]==self.vld and statement[8]=='ACCEPT','signed statement inconsistency')
        wire_pair=(statement[4],statement[5])
        require(len(set(wire_pair))==2 and all(a in AUTHORITIES for a in wire_pair),'invalid token signers')
        msg=encode(*statement)
        require(all(self.pqc.verify(self.sigkeys[a][0],msg,wire_token['signatures'][a]) for a in wire_pair),'EV token signature invalid')
        require((self.sid,self.ac,self.resource.nu) not in self.executed,'token already executed')
        self.stage='EV_ACCEPTED'; self.mark('ev_token_verification', decision='ACCEPT')
    def execute(self,power_w=1200,seconds=30,tick_s=5):
        require(self.stage=='EV_ACCEPTED','EV did not accept token')
        tid=(self.sid,self.ac,self.resource.nu)
        require(tid not in self.executed,'repeat authorization execution')
        self.executed.add(tid)  # atomic authorization consumption modeled by memory store
        nx=secrets.token_bytes(16)
        req=encode('QSV2G-START',self.sid,self.ac,self.resource.nu,self.op,nx)
        pkt=self.e2c.encrypt(req,encode('QSV2G-START',self.sid))
        self.wire_bytes['execution_start']=len(pkt['nonce'])+len(pkt['ciphertext'])
        require(ct_eq(self.e2c.decrypt(pkt,encode('QSV2G-START',self.sid)),req),'start message rejected')
        require(self.vld[0]<=int(time.time())<=self.vld[1],'validity expired before execution')
        # Integer micro-Wh units avoid floating-point serialization or rounding ambiguity.
        energy_uwh=0; limit_uwh=self.limit['max']*1000; steps=0
        logical_time=self.t0
        for _ in range(math.ceil(seconds/tick_s)):
            dt=min(tick_s,seconds-steps*tick_s)
            if dt<=0 or logical_time > self.vld[1]: break
            increment=round(power_w*dt/3600*1_000_000)
            energy_uwh+=min(increment,limit_uwh-energy_uwh)
            steps+=1; logical_time+=dt
            if energy_uwh>=limit_uwh: break
        require(energy_uwh<=limit_uwh,'energy ceiling exceeded')
        self.energy_mwh=energy_uwh/1000
        self.stage='EXECUTED'; self.mark('energy_execution', energy_mwh=self.energy_mwh, ticks=steps, op=self.op)
    def terminate(self, interrupted=False):
        require(self.stage=='EXECUTED','no active or completed execution')
        status='INTERRUPTED' if interrupted else 'COMPLETED'
        record={'sid':self.sid,'ac':self.ac,'nu':self.resource.nu,'op':self.op,
                'status':status,'ended_at':int(time.time()), 'metered_uWh':int(round(self.energy_mwh*1000))}
        ad=encode('QSV2G-TERM',self.sid,self.csid)
        pkt=self.c2e.encrypt(canon(record),ad)
        self.wire_bytes['termination']=len(pkt['nonce'])+len(pkt['ciphertext'])
        require(ct_eq(self.c2e.decrypt(pkt,ad),canon(record)),'termination record rejected')
        self.stage='TERMINATED'; self.mark('termination',status=status)
        return record

def run_demo(crypto=None, pair=('A','B'), seed=2026,L=162,**fault):
    p=QSV2G(crypto=crypto,seed=seed,L=L,available=fault.get('available'))
    try:
        p.register()
        p.authenticate(tamper_ev=fault.get('tamper_ev',False),tamper_cs=fault.get('tamper_cs',False))
        p.establish_session()
        p.authorization_context(op=fault.get('op','CHARGE'),limit_mwh=fault.get('limit_mwh',12000),
                                lifetime_sec=fault.get('lifetime_sec',600),tamper_ciphertext=fault.get('tamper_context',False))
        p.authority_verify(approvals=fault.get('approvals',AUTHORITIES),revoked=fault.get('revoked',False),bad_consent=fault.get('bad_consent',False))
        p.threshold_reconstruct(pair=pair,tamper_share=fault.get('tamper_share',False))
        p.issue_token(tamper_signature=fault.get('tamper_signature',False))
        p.protected_response(tamper_response=fault.get('tamper_response',False),tamper_token_after_collector=fault.get('tamper_token_after_collector',False))
        p.execute()
        p.terminate()
        return p
    except ProtocolAbort as e:
        p.mark(p.stage,'REJECTED',reason=str(e))
        return p
'''
actual_core_hash = hashlib.sha256(CORE_SOURCE.encode('utf-8')).hexdigest()
assert actual_core_hash == REFERENCE_CODE_SHA256, f'CORE SOURCE MISMATCH: {actual_core_hash}'
exec(compile(CORE_SOURCE, 'qsv2g_reference_core.py', 'exec'), globals())
print('Reference implementation SHA-256 VERIFIED:',actual_core_hash)


## 3 — Original test harness

This harness uses real standardized post-quantum cryptography and records **expected**, **observed**, and **match** separately. The suite fails loudly if the implementation deviates from an expected test outcome.

In [ ]:
import pandas as pd, numpy as np, itertools, collections, json, zipfile, importlib.metadata, hashlib, os, time
from IPython.display import display
pqc = PQCrypto()  # raises if real ML-KEM-768 / ML-DSA-65 unavailable
print('Post-quantum cryptographic backend:',pqc.kind)
rows=[]
def record(suite,name,expected,observed,comment='',severity='functional'):
    passed=(expected==observed)
    rows.append({'suite':suite,'case':name,'expected':expected,'observed':observed,'pass':passed,'interpretation':comment,'type':severity})
    if not passed: raise AssertionError(f'{suite}: {name}: expected {expected}, observed {observed}')
def abort_reason(fn):
    try: fn()
    except ProtocolAbort as e: return 'REJECTED',str(e)
    return 'ACCEPTED','No ProtocolAbort raised'
stage_timings=[]
def start_timer(label): return (label,time.perf_counter())
def stop_timer(t):
    label,started=t
    elapsed=round((time.perf_counter()-started)*1000,3)
    stage_timings.append({'stage':label,'wall_ms':elapsed})
    print(label,'finished in',elapsed,'ms')
print('Reference harness ready.')


## A — Ideal QSS, corruption, and malicious-dealer limitations

This is the original experimental code from the previously validated notebook. Test results are written to fresh CSV files. An expected vulnerability is labeled `known_limitation`, not a security success.

In [ ]:
t_run=start_timer('Suite 8')
# Ideal authorized pair recovery with deterministic quantum statevector probabilities.
rng=np.random.default_rng(20261022)
quantum_rows=[]
for pair in itertools.combinations(AUTHORITIES,2):
    for s in range(3):
        for shot in range(80):
            a,b=measure_pair(qss_state(s),pair,rng)
            rec=recover_symbol(pair,a,b)
            quantum_rows.append({'pair':'-'.join(pair),'secret':s,'recovered':rec,'correct':rec==s})
ideal_df=pd.DataFrame(quantum_rows)
assert ideal_df.correct.all()
record('A','Ideal AB/AC/BC qutrit recovery','ACCEPTED','ACCEPTED',f'{len(ideal_df)} / {len(ideal_df)} exact ideal recoveries')

# Test the commitment check on each originating dealer and each possible authorized pair.
for pair in itertools.combinations(AUTHORITIES,2):
    for dealer in AUTHORITIES:
        q=new_qss_resource(L=9)
        q.reserve(b'transaction-context',pair)
        original=q.dealer_secrets[dealer][0]
        q.states[dealer][0]=qss_state((original+1)%3) # corrupt one share group post-commitment
        status,reason=abort_reason(lambda q=q,pair=pair: q.reconstruct(b'transaction-context',pair,rng))
        assert q.state=='CONSUMED'
        record('A',f'Quantum value substitution dealer {dealer} pair {pair}','REJECTED',status,reason)

# Test corruption of already distributed commitment itself.
q=new_qss_resource(L=12); q.reserve(b'AC',('A','B'))
q.commitments['B']=bytes(32)
status,reason=abort_reason(lambda: q.reconstruct(b'AC',('A','B'),rng))
record('A','Commitment substitution','REJECTED',status,reason)

# A dealer can choose an entirely DIFFERENT well-formed secret and matching commitment.
# This is NOT detected by consistency checks; honest-source semantics are an assumption.
q=new_qss_resource(L=10)
q.dealer_secrets['C']=[(s+1)%3 for s in q.dealer_secrets['C']]
q.states['C']=[qss_state(s) for s in q.dealer_secrets['C']]
q.commitments['C']=H('QSV2G-QSS-COM','C',q.nu,q.dealer_secrets['C'])
q.reserve(b'AC',('A','B'))
status,reason=abort_reason(lambda: q.reconstruct(b'AC',('A','B'),rng))
record('A','Malicious dealer replaces valid state and its own commitment','ACCEPTED',status,
       'Expected: a self-consistent but dealer-chosen secret is not rejected; does not prove a malicious-secure VQSS property.', 'known_limitation')

# The written stabilizer checks accept all three logical secret values.
cons=qss_consistency()
stabilizers_all_pass=all(v['x_stabilizer_error']<1e-12 and v['z_stabilizer_error']<1e-12 for v in cons)
record('A','Stabilizers cannot distinguish encoded 0, 1 and 2','NOT_DETECTED',
       'NOT_DETECTED' if stabilizers_all_pass else 'DETECTED',
       'All three valid codewords have the same indicated stabilizer eigenvalues; extra verification is required.', 'known_limitation')

pd.DataFrame(cons).to_csv(OUT/'qss_stabilizer_checks.csv',index=False)
ideal_df.to_csv(OUT/'qss_pair_ideal.csv',index=False)
print('QSS corruption and consistency cases complete')

stop_timer(t_run)


## B — Six workflows, token verification, and signer-trust boundaries

This is the original experimental code from the previously validated notebook. Test results are written to fresh CSV files. An expected vulnerability is labeled `known_limitation`, not a security success.

In [ ]:
t_run=start_timer('Suite 10')
# Normal end-to-end controls for both operation types and all authorized pairs.
controls=[]
for op,pair in itertools.product(('CHARGE','DISCHARGE'),itertools.combinations(AUTHORITIES,2)):
    flow=run_demo(crypto=pqc,pair=pair,L=15,op=op)
    success=flow.stage=='TERMINATED'
    controls.append({'op':op,'pair':'-'.join(pair),'accepted':success,'stage':flow.stage})
    record('B',f'Full protocol {op} / {pair}','ACCEPTED','ACCEPTED' if success else 'REJECTED',
           'All stages executed through final energy-record processing.')
pd.DataFrame(controls).to_csv(OUT/'authorization_controls.csv',index=False)

# Cannot sign a token in the honest state machine before QSS reconstruction.
p=QSV2G(crypto=pqc,L=12); p.register();p.authenticate();p.establish_session();p.authorization_context();p.authority_verify()
status,reason=abort_reason(p.issue_token)
record('B','Honest token issuance before QSS','REJECTED',status,reason)
assert p.resource.state=='UNUSED'

flow=run_demo(crypto=pqc,L=12,approvals=('A',))
record('B','Only one approving authority','REJECTED','REJECTED' if flow.stage!='TERMINATED' else 'ACCEPTED',flow.log[-1].get('reason',''))

# Try to impersonate authority B with authority A's signing key; EV must reject.
p=QSV2G(crypto=pqc,L=12); p.register();p.authenticate();p.establish_session();p.authorization_context();p.authority_verify()
fake_statement=('QSV2G-AUTHZ',p.ac,p.sid,p.resource.nu,'A','B',secrets.token_bytes(32),p.vld,'ACCEPT')
encoded=encode(*fake_statement)
sig_a=pqc.sign(p.sigkeys['A'][1],encoded)
p.token={'statement':fake_statement,'signatures':{'A':sig_a,'B':sig_a}}
p.stage='TOKEN'
status,reason=abort_reason(p.protected_response)
record('B','Single compromised signer impersonates second authority','REJECTED',status,reason)

# Two compromised authorities can sign without using quantum resources.
# We explicitly bypass *their own issuance policy*, not the EV's signature verifier.
p=QSV2G(crypto=pqc,L=12); p.register();p.authenticate();p.establish_session();p.authorization_context();p.authority_verify()
assert p.resource.state=='UNUSED'
fake_statement=('QSV2G-AUTHZ',p.ac,p.sid,p.resource.nu,'A','B',secrets.token_bytes(32),p.vld,'ACCEPT')
encoded=encode(*fake_statement)
p.token={'statement':fake_statement,'signatures':{
    'A':pqc.sign(p.sigkeys['A'][1],encoded),
    'B':pqc.sign(p.sigkeys['B'][1],encoded),
}}
p.stage='TOKEN' # models colluding authorities issuing valid signatures outside honest QSS gate
status,reason=abort_reason(p.protected_response)
record('B','Two colluding signers bypass QSS attestation without reconstruction',
       'ACCEPTED','ACCEPTED' if status=='ACCEPTED' and p.stage=='EV_ACCEPTED' and p.resource.state=='UNUSED' else 'REJECTED',
       'Expected trust-limit demonstration: signature-only EV verification does not prove quantum reconstruction.', 'known_limitation')
print('Authorization tests complete; known trust limitation recorded explicitly.')

stop_timer(t_run)


## C — Equation traceability, full execution trace, assumptions

This is the original experimental code from the previously validated notebook. Test results are written to fresh CSV files. An expected vulnerability is labeled `known_limitation`, not a security success.

In [ ]:
t_run=start_timer('Suite 12')
# Mapping assembled against the revised paper's actual equation labels.
map_entries=[
 ('eq:ev_reg_bell','dense_encode_decode','Bell preparation and 2-qubit statevector encoding','implemented'),
 ('eq:ev_reg_tag','QSV2G.register','HMAC-verified Bell transport','implemented'),
 ('eq:cs_reg_auth','QSV2G.register','Station registration proof','implemented'),
 ('eq:qss_qutrit_encoding','qss_state','Three-qutrit ideal state encoding','implemented'),
 ('eq:qss_pair_recovery','recover_symbol','Two-share classical ternary symbol recovery','implemented'),
 ('eq:qsv2g_qss_commitment','new_qss_resource','Classical hash commitment','implemented'),
 ('eq:qss_state_consistency','qss_consistency','Ideal-state stabilizers; not malicious VQSS','partial'),
 ('eq:ev_auth_proof','QSV2G.authenticate','Online EV challenge/credential tag','implemented'),
 ('eq:cs_auth_proof','QSV2G.authenticate','Online station proof','implemented'),
 ('eq:qsv2g_kem_encaps','QSV2G.establish_session','Actual ML-KEM-768 encapsulation','implemented'),
 ('eq:qsv2g_session_key','QSV2G.establish_session','Directional HKDF traffic keys','implemented'),
 ('eq:qsv2g_context','QSV2G.authorization_context','Canonical context hash','implemented'),
 ('eq:qsv2g_ev_authz_consent','QSV2G.authority_verify','EV consent model with registration secret','implemented'),
 ('eq:qsv2g_authority_decision','QSV2G.authority_verify','Toy authority policy and revocation flags','partial'),
 ('eq:qsv2g_qss_recover_symbol','QSSResource.reconstruct','Quantum sampling and classical recovery','implemented'),
 ('eq:qsv2g_qss_commitment_check','QSSResource.reconstruct','Recovered value hash checks','implemented'),
 ('eq:qsv2g_auth_token','QSV2G.issue_token','Two ML-DSA-65 attestations','implemented'),
 ('eq:qsv2g_response_encryption','QSV2G.protected_response','AES-256-GCM response','implemented'),
 ('eq:qsv2g_ev_context_verify','QSV2G.protected_response','EV context and token checks','implemented'),
 ('eq:qsv2g_execution_request','QSV2G.execute','Authenticated execution-start request','implemented'),
 ('eq:qsv2g_execution_states','QSV2G.execute','In-memory single-use state transition','partial'),
 ('eq:qsv2g_termination_protection','QSV2G.terminate','Authenticated termination record','implemented'),
]
import inspect
# Source methods/classes are available in the notebook namespace, checked without claiming formal equivalence.
def symbol_present(name):
    pieces=name.split('.')
    obj=globals().get(pieces[0])
    if obj is None: return False
    for attr in pieces[1:]:
        if not hasattr(obj,attr): return False
        obj=getattr(obj,attr)
    return True
mapping_df=pd.DataFrame([{'latex_equation':lab,'implementation_symbol':func,'purpose':desc,
                          'coverage':coverage,'symbol_exists':symbol_present(func)}
                         for lab,func,desc,coverage in map_entries])
assert mapping_df.symbol_exists.all()
mapping_df.to_csv(OUT/'equation_to_code_mapping.csv',index=False)

# Capture an actual full protocol trace, including phases, not only the final status.
trace=run_demo(crypto=pqc,L=18,pair=('B','C'),op='DISCHARGE')
trace_df=pd.DataFrame(trace.log)
assert trace.stage=='TERMINATED'
trace_df.to_csv(OUT/'protocol_execution_trace.csv',index=False)
record('C','Equation mapping symbols present','MAPPED','MAPPED' if mapping_df.symbol_exists.all() else 'MISSING',
       f'{len(mapping_df)} source equation/function connections checked; equivalence remains a manual review task.')
record('C','End-to-end trace reaches termination','ACCEPTED','ACCEPTED' if trace.stage=='TERMINATED' else 'REJECTED',
       'Actual state transitions exported to CSV.')

# Record critical assumptions that ARE NOT implemented rather than mislabeling them complete.
assumptions=[
 ('Authenticated confidential UC-to-UC network channels','MODELED','In-process trusted call boundaries; no network stack'),
 ('Physical quantum channels, memory and detector noise','NOT_IMPLEMENTED','27-amplitude simulator + synthetic qutrit-X errors only'),
 ('Arbitrary single-authority outage tolerance','NOT_ESTABLISHED','UC-A and UC-B online required by reference flow'),
 ('Malicious-secure verifiable QSS','NOT_ESTABLISHED','Stabilizers plus commitments are insufficient'),
 ('EV independently verifies quantum reconstruction','NOT_IMPLEMENTED','EV verifies two ML-DSA attestations, not a quantum proof'),
 ('Crash-persistent nonce counters and resource ledger','NOT_IMPLEMENTED','Reference state exists only in-memory'),
 ('Physical power grid hardware and trusted metering','NOT_IMPLEMENTED','Toy energy-controller simulation'),
 ('Independent formal cryptographic protocol verification','NOT_PERFORMED','Software tests cannot replace a security proof'),
]
pd.DataFrame(assumptions,columns=['assumption','status','reason']).to_csv(OUT/'explicit_assumptions.csv',index=False)
display(mapping_df)

stop_timer(t_run)


## D — Resource lifecycle, replay, termination, authority availability

This is the original experimental code from the previously validated notebook. Test results are written to fresh CSV files. An expected vulnerability is labeled `known_limitation`, not a security success.

In [ ]:
t_run=start_timer('Suite 14')
# Double reservation is rejected.
q=new_qss_resource(L=6); q.reserve(b'ctx1',('A','B'))
status,reason=abort_reason(lambda: q.reserve(b'ctx2',('B','C')))
record('D','Competing QSS reservation','REJECTED',status,reason)

# Loss after reservation: fail closed, cannot silently release consumed/possibly measured state.
q=new_qss_resource(L=6);q.reserve(b'ctx1',('A','C'))
status,reason=abort_reason(lambda: q.reserve(b'ctx1',('A','C')))
record('D','Unresolved reserved resource cannot be reassigned','REJECTED',status,
       'In-memory RESERVED state has no recovery protocol; safe but can reduce availability.')

# Reconstructed resource is consumed; failed reconstruction cannot be retried.
q=new_qss_resource(L=6);q.reserve(b'ctx1',('A','C'))
status,reason=abort_reason(lambda: q.reconstruct(b'ctx1',('A','C'),rng,tamper=True))
assert q.state=='CONSUMED'
record('D','Failed reconstruction burns resource','REJECTED',status,reason)
status2,reason2=abort_reason(lambda: q.reserve(b'ctx2',('A','B')))
record('D','Consumed resource cannot be reassigned','REJECTED',status2,reason2)

# AEAD message replay and incorrect AD.
traffic=Traffic(secrets.token_bytes(32));ad=encode('QSV2G-TEST','SID')
pkt=traffic.encrypt(b'v2g message',ad)
assert traffic.decrypt(pkt,ad)==b'v2g message'
status,reason=abort_reason(lambda: traffic.decrypt(pkt,ad))
record('D','Authenticated packet replay','REJECTED',status,reason)

pkt2=traffic.encrypt(b'payload',ad)
status,reason=abort_reason(lambda: traffic.decrypt(pkt2,encode('QSV2G-TEST','other-SID')))
record('D','Associated-data substitution','REJECTED',status,reason)

# New Traffic state with the same key resets its send counter; this is a real persistence hazard.
same_key=secrets.token_bytes(32)
p1=Traffic(same_key).encrypt(b'A',ad)
p2=Traffic(same_key).encrypt(b'B',ad)
nonce_collision=(p1['nonce']==p2['nonce'])
record('D','Reinstantiating traffic with the same key repeats the nonce','REPEATED','REPEATED' if nonce_collision else 'DISTINCT',
       'Expected configuration hazard: session traffic keys must never be restored with reset nonce state.', 'known_limitation')

# Expired authorization cannot execute.
expired=run_demo(crypto=pqc,L=12,lifetime_sec=-1)
record('D','Expired transaction rejected','REJECTED','REJECTED' if expired.stage!='TERMINATED' else 'ACCEPTED',
       expired.log[-1].get('reason',''))

# Replay of an already executed transaction.
p=run_demo(crypto=pqc,L=12)
assert p.stage=='TERMINATED'
status,reason=abort_reason(p.execute)
record('D','Second energy execution rejected','REJECTED',status,reason)

# Simulated authority outages.
for offline,pair in [('A',('B','C')),('B',('A','C')),('C',('A','B'))]:
    online=tuple(a for a in AUTHORITIES if a!=offline)
    p=run_demo(crypto=pqc,L=12,pair=pair,available=online)
    expected='ACCEPTED' if offline=='C' else 'REJECTED'
    observed='ACCEPTED' if p.stage=='TERMINATED' else 'REJECTED'
    record('D',f'UC-{offline} unavailable (full workflow)',expected,observed,
           'QSS has 2-of-3 reconstructability, but the full system relies on online UC-A and UC-B.', 'availability_boundary')

# Protected interruption record (honest channel model).
p=QSV2G(crypto=pqc,L=12);p.register();p.authenticate();p.establish_session();p.authorization_context()
p.authority_verify();p.threshold_reconstruct();p.issue_token();p.protected_response();p.execute()
term=p.terminate(interrupted=True)
record('D','Authenticated interrupted termination','INTERRUPTED',term['status'],'Terminal receipt protected by the directional traffic key.')
print('Failure and recovery checks complete')

stop_timer(t_run)


## 4 — Direct comparison against the submitted validation archive

Compare the new execution to the archived **case names, expected outcomes, observed categories, and suite/type labels**. We intentionally do not compare random credential values, ciphertexts, or elapsed times byte-for-byte. Also verify the source-to-paper equation labels against the pinned `.tex` snapshot; this is traceability, not mathematical equivalence proof.

In [ ]:
t_run=start_timer('Reference comparison')
all_df=pd.DataFrame(rows)
assert all_df['pass'].astype(bool).all(), 'Original suite generated mismatched expected outcomes'
assert len(all_df)==38, f'Unexpected test count: {len(all_df)}'
assert (all_df['type']=='known_limitation').sum()==4
assert len(ideal_df)==720 and bool(ideal_df['correct'].all())
assert len(controls)==6 and all(c['accepted'] and c['stage']=='TERMINATED' for c in controls)
assert trace.stage=='TERMINATED'
assert len(mapping_df)==22 and mapping_df['symbol_exists'].all()
old_df=pd.read_csv(io.BytesIO(REFERENCE_ZIP.read('validation_case_results.csv')))
keys=['suite','case','expected','observed','type']
old=old_df[keys].sort_values(['suite','case']).reset_index(drop=True)
new=all_df[keys].sort_values(['suite','case']).reset_index(drop=True)
assert len(old)==len(new)==38
comparison=new.merge(old,on=['suite','case'],suffixes=('_current','_archived'),how='outer',indicator=True)
for col in ('expected','observed','type'):
    comparison[col+'_match']=(comparison[col+'_current']==comparison[col+'_archived'])
comparison['reproduced']=(comparison['_merge']=='both') & comparison[[c+'_match' for c in ('expected','observed','type')]].all(axis=1)
comparison.to_csv(STAGE_ROOT/'baseline_case_comparison.csv',index=False)
assert len(comparison)==38 and bool(comparison['reproduced'].all()),'New test cases differ from archived results; see baseline_case_comparison.csv'

archived_ideal=pd.read_csv(io.BytesIO(REFERENCE_ZIP.read('qss_pair_ideal.csv')))
assert len(archived_ideal)==720 and archived_ideal['correct'].astype(bool).all()
archived_controls=pd.read_csv(io.BytesIO(REFERENCE_ZIP.read('authorization_controls.csv')))
current_controls=pd.DataFrame(controls)
cols=['op','pair','accepted','stage']
assert current_controls[cols].sort_values(['op','pair']).reset_index(drop=True).equals(archived_controls[cols].sort_values(['op','pair']).reset_index(drop=True))

tex_labels=set(re.findall(r'\\label\{([^}]+)\}',REFERENCE_TEX))
label_checks=mapping_df.copy()
label_checks['latex_label_exists']=label_checks['latex_equation'].map(lambda s:s in tex_labels)
label_checks.to_csv(STAGE_ROOT/'equation_label_checks.csv',index=False)
assert bool(label_checks['latex_label_exists'].all()),'Pinned reference paper is missing a mapped LaTeX label'

all_df.to_csv(OUT/'validation_case_results.csv',index=False)
print('Matched 38/38 case outcome categories against submitted archive')
print('Matched 6/6 authorization workflow results')
print('Verified 720/720 ideal QSS trials (old and new)')
print('Found 22/22 equation labels and implementation symbols')
stop_timer(t_run)


## 5 — Environment record, manifest, strict summary, and evidence ZIP

Files produced here come from **this execution** except the archived comparison fixture (used only as reference). An exact prior outcome match means functional reproducibility for tested cases, not a proof of security or matching wall-clock performance.

In [ ]:
from importlib import metadata
import socket, getpass

def package_version(name):
    try: return metadata.version(name)
    except metadata.PackageNotFoundError: return None

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''): h.update(chunk)
    return h.hexdigest()

environment={
 'run_started_utc':RUN_STARTED_UTC,
 'run_finished_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'execution_location_claim':'kaggle_path_detected' if Path('/kaggle/working').is_dir() else 'local_non_kaggle',
 'python':sys.version,
 'platform':platform.platform(),
 'openssl_version':openssl_ver,
 'backend':pqc.kind,
 'packages':{p:package_version(p) for p in ['numpy','pandas','pqcrypto','cryptography','qiskit','pennylane','nbformat']},
 'randomness':{'numpy_ideal_pair_seed':20261022,'other_protocol_keys':'OS cryptographic randomness; not deterministic'},
 'accelerator_required':False,
 'physical_quantum_hardware_used':False,
}
(STAGE_ROOT/'environment.json').write_text(json.dumps(environment,indent=2,default=str))
(STAGE_ROOT/'source_hashes.json').write_text(json.dumps({
 'reference_core_sha256':actual_core_hash,
 'reference_section_iv_tex_sha256':REFERENCE_TEX_SHA256,
 'archived_submitted_validation_zip_sha256':ARCHIVED_VALIDATION_SHA256,
 'note':'These hashes refer to the earlier tested Section IV revision, not automatically to the latest manuscript.'
},indent=2))
(OUT/'validation_summary.json').write_text(json.dumps({
 'protocol':'QSV2G','stage':'baseline_reproducibility',
 'test_count':int(len(all_df)),
 'expected_outcomes_matched':int(all_df['pass'].sum()),
 'known_limitations_demonstrated':int((all_df['type']=='known_limitation').sum()),
 'qss_ideal_pair_trials':int(len(ideal_df)),
 'qss_ideal_correct':int(ideal_df['correct'].sum()),
 'cryptographic_backend':pqc.kind,
 'source_implementation_sha256':actual_core_hash,
 'reference_tex_sha256':REFERENCE_TEX_SHA256,
 'security_scope':'software behavior; not a proof of malicious-secure QSS or physical quantum deployment'
},indent=2))

summary={
 'stage':'01_baseline_reproducibility',
 'overall':'PASS' if bool(comparison['reproduced'].all()) and len(ideal_df)==720 and all(ideal_df['correct']) else 'FAIL',
 'reference_implementation_sha256_match':actual_core_hash==REFERENCE_CODE_SHA256,
 'reference_tex_sha256_match':hashlib.sha256(TEX_BYTES).hexdigest()==REFERENCE_TEX_SHA256,
 'archived_suite_case_matches':int(comparison['reproduced'].sum()),
 'archived_suite_case_count':int(len(comparison)),
 'software_validation_expected_matches':int(all_df['pass'].sum()),
 'software_validation_cases':int(len(all_df)),
 'known_limitation_demonstrations':int((all_df['type']=='known_limitation').sum()),
 'ideal_qss_correct':int(ideal_df['correct'].sum()),
 'ideal_qss_trials':int(len(ideal_df)),
 'charge_discharge_pair_workflows_terminated':int(current_controls['accepted'].sum()),
 'charge_discharge_pair_workflows_total':int(len(current_controls)),
 'equation_labels_verified':int(label_checks['latex_label_exists'].sum()),
 'equation_labels_total':int(len(label_checks)),
 'backend':pqc.kind,
 'limitations':['ideal quantum simulator, not quantum networking',
   'known malicious dealer, stabilizer and attestation limitations remain',
   'UC-A and UC-B online dependencies remain',
   'key/nonce state restoration hazard remains',
   'equation mapping is traceability, not formal verification',
   'cryptographic entropy means outputs are not bitwise deterministic']
}
(STAGE_ROOT/'baseline_summary.json').write_text(json.dumps(summary,indent=2))
pd.DataFrame(stage_timings).to_csv(STAGE_ROOT/'stage_wall_timings.csv',index=False)

(STAGE_ROOT/'README_RESULTS.txt').write_text("""QSV2G STAGE 01: BASELINE REPRODUCIBILITY
========================================
See baseline_summary.json (main results), environment.json (software provenance),
source_hashes.json (exact reference source), and baseline_case_comparison.csv
(previous submitted 38 cases versus new execution).
The validation/ directory contains NEW outputs from the original 38-case suite.
A result of PASS does not establish malicious-secure QSS or quantum hardware feasibility.
Do not merge wall-clock timings with earlier backends/environments.
Upload this ZIP AND the Kaggle-executed .ipynb to ChatGPT for review.
""")

# Manifest covers all files except itself to avoid recursive hashing.
manifest=[]
for p in sorted(STAGE_ROOT.rglob('*')):
    if p.is_file() and p.name!='artifact_manifest.csv':
        manifest.append({'path':p.relative_to(STAGE_ROOT).as_posix(), 'size_bytes':p.stat().st_size, 'sha256':sha_file(p)})
pd.DataFrame(manifest).to_csv(STAGE_ROOT/'artifact_manifest.csv',index=False)

archive=STAGE_ROOT.parent/'qsv2g_stage01_baseline_results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(STAGE_ROOT.rglob('*')):
        if p.is_file(): z.write(p,arcname=p.relative_to(STAGE_ROOT))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    assert len(z.namelist()) == sum(1 for p in STAGE_ROOT.rglob('*') if p.is_file())
print('STAGE RESULT:',summary['overall'])
print('NEW cases:',len(all_df),'expected matched:',all_df['pass'].sum(),
      '; archival comparison:',int(comparison['reproduced'].sum()),'/',len(comparison))
print('QSS recovery:',int(ideal_df['correct'].sum()),'/',len(ideal_df))
print('RESULT ZIP:',archive.resolve(),'BYTES:',archive.stat().st_size)
print('Store the executed notebook too; original Kaggle run URL provides stronger provenance.')
display(all_df.groupby('suite').agg(cases=('pass','size'),matches=('pass','sum')).reset_index())


## Interpret the result correctly

**PASS** means the specified 38 expected test outcomes were reproduced and the ideal QSS, six workflows, and source-reference checks passed. This is a **software regression and reproducibility result**, not evidence of malicious-secure verifiable QSS, formal resistance to arbitrary quantum adversaries, quantum network deployment, or guaranteed physical metering. The four previously documented known limitations remain limitations even when their tests correctly reproduce them.

**Download from Kaggle Outputs:** `qsv2g_stage01_baseline_results.zip`; also download the **executed notebook `.ipynb`** (with cell outputs). Send both here. If the run aborts, send the executed notebook and full error text. Stage 02 should use this same pinned source unless we explicitly announce and justify a protocol correction.